## 🔰PyTorchでニューラルネットワーク基礎 #26 【トークナイズ編・word piece】

### 内容
* Qiitaの記事と連動しています

### データについて
cc100から日本語の部分を2万行抽出したもの。text列に対応する言語の文書
* 日本語：tiny_cc100_ja.csv
* わかち書き版：tiny_cc100_ja_wakati.csv

### ちょっとした注意点
* **データやトークナイザーの保存先に注意**。適宜修正してください。
* pre_tokenizers.BertPreTokenizer()のみのタイプ
* 利用ライブラリ：tokenizers
* データ数が多い場合はシャッフルする部分を少し変更しないとNG

In [1]:
import random
import pandas as pd
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# (1) 変更点　BPE → WordPiece
# tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer = Tokenizer(models.WordPiece(unk_token="<unk>"))

#(2) 初期分割の方法
# Bertタイプというのがあるので使ってみた。空白のみ置き換えの場合はmetaspace
# utf8（0〜255）の数値を使い場合は、bytelevelで対応します。
#tokenizer.pre_tokenizer = pre_tokenizers.Metaspace(replacement="▁")
tokenizer.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
#tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)


# (3) 変更点　BpeTrainer → WordPieceTrainer
trainer = trainers.WordPieceTrainer(
    vocab_size=15_000,
    special_tokens=["<pad>", "<bos>", "<eos>", "<unk>", "<mask>"],
    min_frequency=2
)


# (4) csvファイルから直接学
paths = ["./data/tiny_cc100_ja.csv"]         # キーにtextがあるテキストファイル
#paths = ["./data/tiny_cc100_ja_wakati.csv"]  # 分かち書きしてみた

# (5) ランダムにしなくてもいいけど、面倒なので前回の多言語仕様をそのまま使ってしまった
def mixed_iterator(paths):
    texts = []
    for p in paths:
        # text列だけ読み込む
        df = pd.read_csv(p)
        texts.extend(df["text"].tolist())   
    # 一気にシャッフル（数百万件程度までならこの方法でOKなはず）
    random.shuffle(texts)  
    for t in texts:
        yield t

# (6) 学習
tokenizer.train_from_iterator(mixed_iterator(paths), trainer=trainer)

# (7) トークナイザーを保存　語彙集合
tokenizer.save("./tokenizer/tiny_word_piece_tokenizer.json")

In [2]:
# 保存したトークナイザーで確認
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file("./tokenizer/tiny_word_piece_tokenizer.json")

print("特殊トークンID:")
print(f"<pad>: {tokenizer.token_to_id('<pad>')}")
print(f"<bos>: {tokenizer.token_to_id('<bos>')}")
print(f"<eos>: {tokenizer.token_to_id('<eos>')}")
print(f"<unk>: {tokenizer.token_to_id('<unk>')}")
print(f"<mask>: {tokenizer.token_to_id('<mask>')}")
print(f"size: {tokenizer.get_vocab_size()}")


text_list = [
    "これは日本語のテストです",
    "これは日本語のテストです🍀",
    "これは日本語のテストです 🍀",
    "你好",
]

for text in text_list:
    encoded = tokenizer.encode(text)
    print(f"文章: {text}")
    print("トークン:", encoded.tokens)
    print("ID:", encoded.ids)
    print(f"デコード: {tokenizer.decode(encoded.ids)}\n")

特殊トークンID:
<pad>: 0
<bos>: 1
<eos>: 2
<unk>: 3
<mask>: 4
size: 15000
文章: これは日本語のテストです
トークン: ['これは', '##日本語', '##の', '##テスト', '##です']
ID: [5847, 8192, 2799, 10111, 5414]
デコード: これは ##日本語 ##の ##テスト ##です

文章: これは日本語のテストです🍀
トークン: ['<unk>']
ID: [3]
デコード: 

文章: これは日本語のテストです 🍀
トークン: ['これは', '##日本語', '##の', '##テスト', '##です', '<unk>']
ID: [5847, 8192, 2799, 10111, 5414, 3]
デコード: これは ##日本語 ##の ##テスト ##です

文章: 你好
トークン: ['<unk>']
ID: [3]
デコード: 



In [ ]:
# 保存したトークナイザーで確認
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file("./tokenizer/tiny_word_piece_tokenizer.json")

print("特殊トークンID:")
print(f"<pad>: {tokenizer.token_to_id('<pad>')}")
print(f"<bos>: {tokenizer.token_to_id('<bos>')}")
print(f"<eos>: {tokenizer.token_to_id('<eos>')}")
print(f"<unk>: {tokenizer.token_to_id('<unk>')}")
print(f"<mask>: {tokenizer.token_to_id('<mask>')}")
print(f"size: {tokenizer.get_vocab_size()}")


text_list = [
    "これは日本語のテストです",
    "これは日本語のテストです🍀",
    "これは日本語のテストです 🍀",
    "你好",
]

for text in text_list:
    encoded = tokenizer.encode(text)
    print(f"文章: {text}")
    print("トークン:", encoded.tokens)
    print("ID:", encoded.ids)
    print(f"デコード: {tokenizer.decode(encoded.ids)}\n")

特殊トークンID:
<pad>: 0
<bos>: 1
<eos>: 2
<unk>: 3
<mask>: 4
size: 15000
文章: これは日本語のテストです
トークン: ['これは', '##日本語', '##の', '##テスト', '##です']
ID: [5847, 8192, 2813, 10022, 5414]
デコード: これは ##日本語 ##の ##テスト ##です

文章: これは日本語のテストです🍀
トークン: ['<unk>']
ID: [3]
デコード: 

文章: これは日本語のテストです 🍀
トークン: ['これは', '##日本語', '##の', '##テスト', '##です', '<unk>']
ID: [5847, 8192, 2813, 10022, 5414, 3]
デコード: これは ##日本語 ##の ##テスト ##です

文章: 你好
トークン: ['<unk>']
ID: [3]
デコード: 

